In [0]:
%pip install faker

In [0]:
%restart_python

In [0]:
%run "../config/schema_configs"

In [0]:
import time
import random
from datetime import datetime, timedelta
from faker import Faker
from pyspark.sql import Row

fake = Faker()

# Limpeza de execuções de arquivos antigas 
try:
    dbutils.fs.rm(LANDING_BASE, recurse=True)
    print(" - Diretórios antigos de arquivos limpos.")
except:
    pass

# função de sazonalidade temporal
def calculcar_timestamp_sazonal(batch_id):
    data_atual = datetime.now()
    dias_retroativos = max(0, 30 - int(batch_id / 2))

    if random.random() > 0.25:
        hora = random.randint(10, 22) # horário de pico das 10h as 22h
    else:
        hora = random.randint(0, 23) # horário fora do pico das 0h as 23h

    minuto = random.randint(0, 59)
    segundo = random.randint(0, 59)

    data_sazonal = data_atual - timedelta(days=dias_retroativos)
    data_sazonal = data_sazonal.replace(hour=hora, minute=minuto, second=segundo)

    return data_sazonal.isoformat()

def generate_clickstream_batch(batch_id, num_records=100):
    # gera um arquivo JSON complexo simulando eventos de cliques no site
    events = []
    actions = ["view_product", "add_to_cart", "remove_from_cart", "click_banner"]

    for _ in range(num_records):
        # introduz anomalias propositais
        user_id = f"USR_{random.randint(1000, 9999)}" if random.random() > 0.02 else None

        event = {
            "event_id": f"EVT_{fake.uuid4()[:8].upper()}",
            "user_id": user_id,
            "timestamp": calculcar_timestamp_sazonal(batch_id),
            "action": random.choice(actions),
            "device_info": {
                "os": random.choice(["iOS", "Android", "Windows", "MacOS"]),
                "browser": random.choice(["Safari", "Chrome", "Firefox"])
            },
            "payload": {
                "product_id": f"PROD_{random.randint(100, 500)}",
                "session_id": f"SESS_{random.randint(50000, 99999)}"
            }
        }

        # Simulação de Schema Evolution: A partir do lote 3, um novo campo surge no JSON
        if batch_id >= 3:
            event["payload"]["discount_applied"] = random.choice([True, False])
            
        events.append(Row(**event))   

    df = spark.createDataFrame(events)  

    # define o nome do aqruivo JSON e salva no volume de landing zone
    timestamp_str = datetime.now().strftime("%Y%m%d_%H%m%S")
    path_destino = f"{BRONZE_CONFIG['clickstream']['source_path']}/batch_{batch_id}_{timestamp_str}"
    df.coalesce(1).write.json(path_destino)

def generate_sales_batch(batch_id, num_records=20):
    # gera dados de vendas estruturados em CSV.
    sales = []
    for _ in range(num_records):
        # Simulação de erro de qualidade: transaction_id nulo para testar a Quarentena
        tx_id = f"TX_{random.randint(10000, 99999)}" if random.random() > 0.05 else None
        
        sale = {
            "transaction_id": tx_id,
            "user_id": f"USR_{random.randint(1000, 9999)}",
            "product_id": f"PROD_{random.randint(100, 500)}",
            "amount": round(random.uniform(10.0, 500.0), 2),
            "payment_method": random.choice(["credit_card", "pix", "boleto"]),
            "transaction_timestamp": calculcar_timestamp_sazonal(batch_id)
        }
        sales.append(Row(**sale))
        
    df = spark.createDataFrame(sales)

    # define o nome do aqruivo JSON e salva no volume de landing zone
    timestamp_str = datetime.now().strftime("%Y%m%d_%H%m%S")
    path_destino = f"{BRONZE_CONFIG['sales']['source_path']}/batch_{batch_id}_{timestamp_str}"
    df.coalesce(1).write.json(path_destino)

# --- LOOP DE SIMULAÇÃO EM TEMPO REAL ---
# roda 5 iterações simulando a chegada de novos arquivos a cada 5 segundos
print("Iniciando a geração de dados simulada...")
for batch in range(1, 61):
    print(f"Gerando lote {batch}/60...")
    generate_clickstream_batch(batch_id=batch, num_records=500)
    generate_sales_batch(batch_id=batch, num_records=100)
    
    # Gerando duplicidade intencional no lote 2 para testar deduplicação na Silver
    if batch in (10, 20, 30, 40, 50):
#        print("⚡ Injetando dados duplicados para testar resiliência...")
        generate_clickstream_batch(batch_id=batch, num_records=30) 
        
    time.sleep(5)

print("Geração concluída com sucesso! Os dados estão prontos na pasta landing.")              